# Phase 4: NLP Medical Report Generator + FAISS RAG

In this phase, we create a local medical knowledge base, retrieve relevant information using FAISS, and generate a structured educational medical report using fixed report templates.


In [1]:
import importlib.util

packages = ["faiss", "sentence_transformers"]

for package in packages:
    status = "Installed" if importlib.util.find_spec(package) else "Not installed"
    print(package, ":", status)

faiss : Installed
sentence_transformers : Installed


In [111]:
%pip install faiss-cpu sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Import Libraries

import os
import json #Convert human readable data to machine readable data - eg. word file into python dictionary, list, etc.
import numpy as np
import faiss #Library for embedding search with vector databases

from sentence_transformers import SentenceTransformer #Convert text to vector embeddings

/home/arslan/projects/skin-lesion-classifier/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Project Paths**

In [113]:
project_path = "/home/arslan/projects/skin-lesion-classifier"

documents_path = project_path + "/rag_data/documents"
index_path = project_path + "/rag_data/index"

os.makedirs(documents_path, exist_ok=True)
os.makedirs(index_path, exist_ok=True)

print("RAG folders are ready.")

RAG folders are ready.


**Lesion Class-Name Mapping**

In [114]:
class_names = {
    "akiec": "Actinic Keratoses and Intraepithelial Carcinoma",
    "bcc": "Basal Cell Carcinoma",
    "bkl": "Benign Keratosis-like Lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic Nevi",
    "vasc": "Vascular Lesions"
}

print(class_names)

{'akiec': 'Actinic Keratoses and Intraepithelial Carcinoma', 'bcc': 'Basal Cell Carcinoma', 'bkl': 'Benign Keratosis-like Lesions', 'df': 'Dermatofibroma', 'mel': 'Melanoma', 'nv': 'Melanocytic Nevi', 'vasc': 'Vascular Lesions'}


**Knowledge-Base Structure**

In [115]:
knowledge_base = []

for code, name in class_names.items():
    knowledge_base.append({
        "class_code": code,
        "class_name": name,
        "description": "",
        "typical_signs": "",
        "risk_level": "",
        "recommended_action": "",
        "source": ""
    })

print("Knowledge base entries:", len(knowledge_base))

Knowledge base entries: 7


**Medical Knowledge for akiec**

In [116]:
knowledge_base[0].update({
    "description": "A rough, scaly skin growth caused mainly by long-term sun exposure.",
    "typical_signs": "Dry, rough, scaly, red, pink or brown patch on sun-exposed skin.",
    "risk_level": "Moderate — it is precancerous and may develop into skin cancer.",
    "recommended_action": "Arrange a dermatologist examination for proper diagnosis and treatment.",
  "source": "National Cancer Institute and American Academy of Dermatology"
})
print(json.dumps(knowledge_base[0], indent=5
))

{
     "class_code": "akiec",
     "class_name": "Actinic Keratoses and Intraepithelial Carcinoma",
     "description": "A rough, scaly skin growth caused mainly by long-term sun exposure.",
     "typical_signs": "Dry, rough, scaly, red, pink or brown patch on sun-exposed skin.",
     "risk_level": "Moderate \u2014 it is precancerous and may develop into skin cancer.",
     "recommended_action": "Arrange a dermatologist examination for proper diagnosis and treatment.",
     "source": "National Cancer Institute and American Academy of Dermatology"
}


**Medical Knowledge for bcc**

In [117]:
knowledge_base[1].update({
    "description": "Basal cell carcinoma is a common skin cancer that begins in the lower part of the epidermis.",
    "typical_signs": "A slow-growing pearly bump, pink patch, bleeding sore, or sore that does not heal.",
    "risk_level": "High — it is cancerous, but it rarely spreads to other parts of the body.",
    "recommended_action": "Arrange a dermatologist examination as soon as possible for diagnosis and treatment.",
    "source": "National Cancer Institute and American Academy of Dermatology"
})

print(json.dumps(knowledge_base[1], indent=4))

{
    "class_code": "bcc",
    "class_name": "Basal Cell Carcinoma",
    "description": "Basal cell carcinoma is a common skin cancer that begins in the lower part of the epidermis.",
    "typical_signs": "A slow-growing pearly bump, pink patch, bleeding sore, or sore that does not heal.",
    "risk_level": "High \u2014 it is cancerous, but it rarely spreads to other parts of the body.",
    "recommended_action": "Arrange a dermatologist examination as soon as possible for diagnosis and treatment.",
    "source": "National Cancer Institute and American Academy of Dermatology"
}


**Medical Knowledge for bkl**

In [118]:
knowledge_base[2].update({
    "description": "Benign keratosis-like lesions are non-cancerous skin growths such as seborrheic keratosis, solar lentigo and lichen planus-like keratosis.",
    "typical_signs": "A brown, tan or dark patch or raised growth that may look waxy, rough, scaly or stuck onto the skin.",
    "risk_level": "Low — these lesions are usually benign, but they can sometimes resemble skin cancer.",
    "recommended_action": "Consult a dermatologist if the lesion changes, grows quickly, bleeds, itches or looks unusual.",
    "source": "HAM10000 dataset paper and American Academy of Dermatology"
})

print(json.dumps(knowledge_base[2], indent=4))

{
    "class_code": "bkl",
    "class_name": "Benign Keratosis-like Lesions",
    "description": "Benign keratosis-like lesions are non-cancerous skin growths such as seborrheic keratosis, solar lentigo and lichen planus-like keratosis.",
    "typical_signs": "A brown, tan or dark patch or raised growth that may look waxy, rough, scaly or stuck onto the skin.",
    "risk_level": "Low \u2014 these lesions are usually benign, but they can sometimes resemble skin cancer.",
    "recommended_action": "Consult a dermatologist if the lesion changes, grows quickly, bleeds, itches or looks unusual.",
    "source": "HAM10000 dataset paper and American Academy of Dermatology"
}


**Medical Knowledge for df**

In [119]:
knowledge_base[3].update({
    "description": "Dermatofibroma is a common benign skin growth made of fibrous tissue.",
    "typical_signs": "A small, firm pink, tan or brown bump that may dimple inward when pinched.",
    "risk_level": "Low — it is usually harmless and does not become cancerous.",
    "recommended_action": "Consult a dermatologist if it grows, changes, bleeds, becomes painful or looks unusual.",
    "source": "DermNet and British Association of Dermatologists"
})

print(json.dumps(knowledge_base[3], indent=4))

{
    "class_code": "df",
    "class_name": "Dermatofibroma",
    "description": "Dermatofibroma is a common benign skin growth made of fibrous tissue.",
    "typical_signs": "A small, firm pink, tan or brown bump that may dimple inward when pinched.",
    "risk_level": "Low \u2014 it is usually harmless and does not become cancerous.",
    "recommended_action": "Consult a dermatologist if it grows, changes, bleeds, becomes painful or looks unusual.",
    "source": "DermNet and British Association of Dermatologists"
}


**Medical Knowledge for mel**

In [120]:
knowledge_base[4].update({
    "description": "Melanoma is a serious skin cancer that begins in melanocytes, the cells that produce skin pigment.",
    "typical_signs": "A new or changing mole with asymmetry, irregular borders, uneven colors, increasing size or continued change.",
    "risk_level": "Very high — melanoma can invade nearby tissue and spread to other parts of the body.",
    "recommended_action": "Arrange an urgent dermatologist examination. A suspicious lesion requires professional assessment and possible biopsy.",
    "source": "National Cancer Institute and American Academy of Dermatology"
})

print(json.dumps(knowledge_base[4], indent=4))

{
    "class_code": "mel",
    "class_name": "Melanoma",
    "description": "Melanoma is a serious skin cancer that begins in melanocytes, the cells that produce skin pigment.",
    "typical_signs": "A new or changing mole with asymmetry, irregular borders, uneven colors, increasing size or continued change.",
    "risk_level": "Very high \u2014 melanoma can invade nearby tissue and spread to other parts of the body.",
    "recommended_action": "Arrange an urgent dermatologist examination. A suspicious lesion requires professional assessment and possible biopsy.",
    "source": "National Cancer Institute and American Academy of Dermatology"
}


**Medical Knowledge for nv**

In [121]:
knowledge_base[5].update({
    "description": "Melanocytic nevi are common moles formed by groups of pigment-producing skin cells.",
    "typical_signs": "A small, round or oval spot that is usually pink, tan or brown with a clear border.",
    "risk_level": "Low — most common moles are benign, but unusual or changing moles may require examination.",
    "recommended_action": "Consult a dermatologist if the mole changes in size, shape, color, sensation, or begins itching or bleeding.",
    "source": "National Cancer Institute and American Academy of Dermatology"
})

print(json.dumps(knowledge_base[5], indent=4))

{
    "class_code": "nv",
    "class_name": "Melanocytic Nevi",
    "description": "Melanocytic nevi are common moles formed by groups of pigment-producing skin cells.",
    "typical_signs": "A small, round or oval spot that is usually pink, tan or brown with a clear border.",
    "risk_level": "Low \u2014 most common moles are benign, but unusual or changing moles may require examination.",
    "recommended_action": "Consult a dermatologist if the mole changes in size, shape, color, sensation, or begins itching or bleeding.",
    "source": "National Cancer Institute and American Academy of Dermatology"
}


**Medical Knowledge for vasc**

In [122]:
knowledge_base[6].update({
    "description": "Vascular lesions are skin growths or abnormalities involving blood vessels, including angiomas, angiokeratomas and pyogenic granulomas.",
    "typical_signs": "A red, purple, blue or dark spot or raised bump that may sometimes bleed.",
    "risk_level": "Low to moderate — many vascular lesions are benign, but some may resemble other skin conditions.",
    "recommended_action": "Consult a dermatologist if the lesion changes, grows quickly, bleeds repeatedly or looks unusual.",
    "source": "HAM10000 dataset paper and DermNet"
})

print(json.dumps(knowledge_base[6], indent=4))

{
    "class_code": "vasc",
    "class_name": "Vascular Lesions",
    "description": "Vascular lesions are skin growths or abnormalities involving blood vessels, including angiomas, angiokeratomas and pyogenic granulomas.",
    "typical_signs": "A red, purple, blue or dark spot or raised bump that may sometimes bleed.",
    "risk_level": "Low to moderate \u2014 many vascular lesions are benign, but some may resemble other skin conditions.",
    "recommended_action": "Consult a dermatologist if the lesion changes, grows quickly, bleeds repeatedly or looks unusual.",
    "source": "HAM10000 dataset paper and DermNet"
}


**Medical Knowledge Base as JSON**

In [123]:
knowledge_file = documents_path + "/skin_lesion_knowledge.json"

with open(knowledge_file, "w") as file:
    json.dump(knowledge_base, file, indent=4)

print("Knowledge base saved successfully.")

Knowledge base saved successfully.


**Load Text-Embedding Model**

In [124]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1259.64it/s]


Embedding model loaded successfully.


**Prepare Knowledge-Base Text for Embeddings**

In [125]:
documents = []

for item in knowledge_base:
    text = (
        f"Class: {item['class_name']}. "
        f"Description: {item['description']} "
        f"Typical signs: {item['typical_signs']} "
        f"Risk level: {item['risk_level']} "
        f"Recommended action: {item['recommended_action']}"
    )
    documents.append(text)

print("Documents prepared:", len(documents))

Documents prepared: 7


**Convert Documents into Embeddings**

In [126]:
embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (7, 384)


**Create the FAISS Index**

In [127]:
embedding_dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(embeddings)

print("Documents stored in FAISS:", faiss_index.ntotal)

Documents stored in FAISS: 7


**Save the FAISS Index**

In [128]:
faiss_file = index_path + "/skin_lesion_faiss.index"

faiss.write_index(faiss_index, faiss_file)

print("FAISS index saved successfully.")

FAISS index saved successfully.


**Test FAISS Retrieval**

In [129]:
query = "What are the signs and risk of melanoma?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = faiss_index.search(query_embedding, k=3)

for rank, index in enumerate(indices[0], start=1):
    print(rank, "-", knowledge_base[index]["class_name"])

1 - Melanoma
2 - Melanocytic Nevi
3 - Actinic Keratoses and Intraepithelial Carcinoma


**Create Reusable FAISS Retrieval Function**

In [130]:
def retrieve_knowledge(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, k=top_k)

    results = []

    for score, index in zip(scores[0], indices[0]):
        result = knowledge_base[index].copy()
        result["similarity_score"] = float(score)
        results.append(result)

    return results

print("Retrieval function created successfully.")

Retrieval function created successfully.


**Test Retrieval Function**

In [131]:
results = retrieve_knowledge(
    "Tell me about melanoma signs and risk",
    top_k=3
)

for result in results:
    print(result["class_name"])
    print("Score:", round(result["similarity_score"], 4))
    print()

Melanoma
Score: 0.6282

Melanocytic Nevi
Score: 0.5437

Actinic Keratoses and Intraepithelial Carcinoma
Score: 0.4927



**Load Primary MobileNetV2 Model**

In [132]:
import tensorflow as tf

model_path = project_path + "/models/mobilenet/mobilenetv2_model.keras"

mobilenet_model = tf.keras.models.load_model(
    model_path,
    compile=False
)

print("Frozen MobileNetV2 model loaded successfully.")

Frozen MobileNetV2 model loaded successfully.


**Import Image Preprocessing Tools**

In [133]:
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMAGE_SIZE = (224, 224)

print("Image preprocessing tools are ready.")

Image preprocessing tools are ready.


**Create Image Preprocessing Function**

In [134]:
def prepare_image(image_path):
    image = load_img(image_path, target_size=IMAGE_SIZE)
    image = img_to_array(image)
    image = np.expand_dims(image, axis=0)
    image = preprocess_input(image)

    return image

print("Image preprocessing function created successfully.")

Image preprocessing function created successfully.


**Create Model Class Order**

In [135]:
class_codes = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc"
]

print("Class order:", class_codes)

Class order: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


**Create Top-3 Prediction Function**

In [136]:
def predict_top3(image_path):
    image = prepare_image(image_path)

    with tf.device("/CPU:0"):
        probabilities = mobilenet_model(
            image,
            training=False
        ).numpy()[0]

    top_indices = np.argsort(probabilities)[-3:][::-1]
    top_predictions = []

    for index in top_indices:
        code = class_codes[index]

        top_predictions.append({
            "class_code": code,
            "class_name": class_names[code],
            "probability": float(probabilities[index])
        })

    return top_predictions

print("Top-3 prediction function restored successfully.")

Top-3 prediction function restored successfully.


**Load Sample Test Image Path**

In [137]:
import pandas as pd

test_df = pd.read_csv(
    project_path + "/data/processed/test.csv"
)

sample_image_path = test_df.iloc[0]["image_path"]

print("Sample image:", sample_image_path)

Sample image: /home/arslan/projects/skin-lesion-classifier/data/raw/HAM10000_images/ISIC_0024316.jpg


**Test The Top-3 Prediction Function**

In [138]:
top_predictions = predict_top3(sample_image_path)

for prediction in top_predictions:
    print(prediction["class_name"])
    print("Probability:", round(prediction["probability"] * 100, 2), "%")
    print()

Melanocytic Nevi
Probability: 86.53 %

Dermatofibroma
Probability: 7.45 %

Actinic Keratoses and Intraepithelial Carcinoma
Probability: 2.37 %



**Connect Top Prediction With FAISS Retrieval**

In [139]:
predicted_class = top_predictions[0]["class_name"]

query = (
    f"Medical information, typical signs, risk level, "
    f"and recommended action for {predicted_class}"
)

retrieved_information = retrieve_knowledge(query, top_k=1)[0]

print(json.dumps(retrieved_information, indent=4))

{
    "class_code": "nv",
    "class_name": "Melanocytic Nevi",
    "description": "Melanocytic nevi are common moles formed by groups of pigment-producing skin cells.",
    "typical_signs": "A small, round or oval spot that is usually pink, tan or brown with a clear border.",
    "risk_level": "Low \u2014 most common moles are benign, but unusual or changing moles may require examination.",
    "recommended_action": "Consult a dermatologist if the mole changes in size, shape, color, sensation, or begins itching or bleeding.",
    "source": "National Cancer Institute and American Academy of Dermatology",
    "similarity_score": 0.6538226008415222
}


**Create Structured Medical Report Function**

In [140]:
def generate_medical_report(top_predictions, retrieved_information):
    top3_text = ""

    for prediction in top_predictions:
        top3_text += (
            f"- {prediction['class_name']}: "
            f"{prediction['probability'] * 100:.2f}%\n"
        )

    report = f"""
SKIN LESION ANALYSIS REPORT

Predicted Lesion: {top_predictions[0]['class_name']}

Top-3 Predictions:
{top3_text}

Description:
{retrieved_information['description']}

Typical Signs:
{retrieved_information['typical_signs']}

Risk Level:
{retrieved_information['risk_level']}

Recommended Action:
{retrieved_information['recommended_action']}

Supporting Source:
{retrieved_information['source']}

Medical Disclaimer:
This system is for educational purposes only and is not a replacement
for diagnosis or advice from a qualified dermatologist.
"""

    return report

print("Medical report function created successfully.")

Medical report function created successfully.


**Generate The Final Medical Report**

In [141]:
medical_report = generate_medical_report(
    top_predictions,
    retrieved_information
)

print(medical_report)


SKIN LESION ANALYSIS REPORT

Predicted Lesion: Melanocytic Nevi

Top-3 Predictions:
- Melanocytic Nevi: 86.53%
- Dermatofibroma: 7.45%
- Actinic Keratoses and Intraepithelial Carcinoma: 2.37%


Description:
Melanocytic nevi are common moles formed by groups of pigment-producing skin cells.

Typical Signs:
A small, round or oval spot that is usually pink, tan or brown with a clear border.

Risk Level:
Low — most common moles are benign, but unusual or changing moles may require examination.

Recommended Action:
Consult a dermatologist if the mole changes in size, shape, color, sensation, or begins itching or bleeding.

Supporting Source:
National Cancer Institute and American Academy of Dermatology

Medical Disclaimer:
This system is for educational purposes only and is not a replacement
for diagnosis or advice from a qualified dermatologist.



**Complete Phase 4 Pipeline Function**

In [142]:
def run_phase4_pipeline(image_path):
    top_predictions = predict_top3(image_path)

    predicted_class = top_predictions[0]["class_name"]

    query = (
        f"Medical information, typical signs, risk level, "
        f"and recommended action for {predicted_class}"
    )

    retrieved_information = retrieve_knowledge(
        query,
        top_k=1
    )[0]

    report = generate_medical_report(
        top_predictions,
        retrieved_information
    )

    return report

print("Complete Phase 4 pipeline created successfully.")

Complete Phase 4 pipeline created successfully.


**Test Complete Phase 4 Pipeline**

In [143]:
final_report = run_phase4_pipeline(sample_image_path)

print(final_report)


SKIN LESION ANALYSIS REPORT

Predicted Lesion: Melanocytic Nevi

Top-3 Predictions:
- Melanocytic Nevi: 86.53%
- Dermatofibroma: 7.45%
- Actinic Keratoses and Intraepithelial Carcinoma: 2.37%


Description:
Melanocytic nevi are common moles formed by groups of pigment-producing skin cells.

Typical Signs:
A small, round or oval spot that is usually pink, tan or brown with a clear border.

Risk Level:
Low — most common moles are benign, but unusual or changing moles may require examination.

Recommended Action:
Consult a dermatologist if the mole changes in size, shape, color, sensation, or begins itching or bleeding.

Supporting Source:
National Cancer Institute and American Academy of Dermatology

Medical Disclaimer:
This system is for educational purposes only and is not a replacement
for diagnosis or advice from a qualified dermatologist.



**Verify Saved RAG Files**

In [144]:
print("Knowledge base exists:", os.path.exists(knowledge_file))
print("FAISS index exists:", os.path.exists(faiss_file))

Knowledge base exists: True
FAISS index exists: True


# User Document RAG System


**Check Document Processing Libraries**

In [145]:
import importlib.util

packages = {
    "pypdf": "PDF files",
    "docx": "Word files"
}

for package, file_type in packages.items():
    status = "Installed" if importlib.util.find_spec(package) else "Not installed"
    print(file_type, ":", status)

PDF files : Installed
Word files : Installed


**Install Document Processing Libraries**

In [146]:
%pip install pypdf python-docx

Note: you may need to restart the kernel to use updated packages.


**Import Document Processing Libraries**

In [147]:
from pypdf import PdfReader
from docx import Document

print("Document processing libraries imported successfully.")

Document processing libraries imported successfully.


**Create Document Text Extraction Function**

In [148]:
def extract_document_text(file_path):
    lower_path = file_path.lower()

    if lower_path.endswith(".pdf"):
        reader = PdfReader(file_path)
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif lower_path.endswith(".docx"):
        document = Document(file_path)
        text = "\n".join(
            paragraph.text
            for paragraph in document.paragraphs
        )

    elif lower_path.endswith(".txt"):
        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as file:
            text = file.read()

    else:
        raise ValueError(
            "Only PDF, DOCX and TXT files are supported."
        )

    if not text.strip():
        raise ValueError(
            "No readable text was found. "
            "Scanned PDFs without selectable text are not supported."
        )

    return text

print("Document text extraction function updated successfully.")

Document text extraction function updated successfully.


**Create Document Chunking Function**

In [149]:
def split_text_into_chunks(text, chunk_size=500):
    words = text.split()
    chunks = []

    for start in range(0, len(words), chunk_size):
        chunk = " ".join(words[start:start + chunk_size])
        chunks.append(chunk)

    return chunks

print("Document chunking function created successfully.")

Document chunking function created successfully.


Splits a long uploaded document into smaller text chunks so they can be converted into embeddings and searched.

**Create Document FAISS Index**

In [150]:
def build_document_index(file_path):
    document_text = extract_document_text(file_path).strip()

    if not document_text:
        raise ValueError("No readable text was found in the document.")

    document_chunks = split_text_into_chunks(document_text)

    document_embeddings = embedding_model.encode(
        document_chunks,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    document_index = faiss.IndexFlatIP(
        document_embeddings.shape[1]
    )

    document_index.add(document_embeddings)

    return document_chunks, document_index

print("Document index function created successfully.")

Document index function created successfully.


**Create Document Retrieval Function**

In [151]:
def retrieve_from_document(
    query,
    document_chunks,
    document_index,
    top_k=3
):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    top_k = min(top_k, len(document_chunks))

    scores, indices = document_index.search(
        query_embedding,
        k=top_k
    )

    results = []

    for score, index in zip(scores[0], indices[0]):
        results.append({
            "text": document_chunks[index],
            "similarity_score": float(score)
        })

    return results

print("Document retrieval function created successfully.")

Document retrieval function created successfully.


**Format Retrieved Document Information**

In [152]:
def format_document_results(document_results):
    if not document_results:
        return "No additional document was uploaded."

    formatted_text = ""

    for number, result in enumerate(document_results, start=1):
        formatted_text += (
            f"Document Section {number}:\n"
            f"{result['text']}\n\n"
        )

    return formatted_text.strip()

print("Document result formatting function created successfully.")

Document result formatting function created successfully.


**Create Extended Medical Report Function**

In [153]:
def generate_extended_medical_report(
    top_predictions,
    retrieved_information,
    user_question="",
    document_results=None
):
    top3_text = ""

    for prediction in top_predictions:
        top3_text += (
            f"- {prediction['class_name']}: "
            f"{prediction['probability'] * 100:.2f}%\n"
        )

    question_text = (
        user_question.strip()
        if user_question.strip()
        else "No additional question was provided."
    )

    document_text = format_document_results(
        document_results
    )

    report = f"""
SKIN LESION ANALYSIS REPORT

Predicted Lesion:
{top_predictions[0]['class_name']}

Top-3 Predictions:
{top3_text}

User Question:
{question_text}

Description:
{retrieved_information['description']}

Typical Signs:
{retrieved_information['typical_signs']}

Risk Level:
{retrieved_information['risk_level']}

Recommended Action:
{retrieved_information['recommended_action']}

Uploaded Document Information:
{document_text}

Supporting Knowledge Source:
{retrieved_information['source']}

Medical Disclaimer:
This system is for educational purposes only and is not a replacement
for diagnosis, treatment or advice from a qualified dermatologist.
Information extracted from an uploaded document may be incomplete
and should also be reviewed by a medical professional.
"""

    return report

print("Extended medical report function created successfully.")

Extended medical report function created successfully.


**Create Complete Image, Question And Document Pipeline**

In [154]:
def run_pipeline_with_document(
    image_path,
    user_question="",
    document_path=None
):
    top_predictions = predict_top3(image_path)

    predicted_class = top_predictions[0]["class_name"]

    query = (
        f"Medical information, typical signs, risk level, "
        f"and recommended action for {predicted_class}. "
        f"{user_question}"
    )

    retrieved_information = retrieve_knowledge(
        query,
        top_k=1
    )[0]

    document_results = []

    if document_path:
        document_chunks, document_index = (
            build_document_index(document_path)
        )

        document_query = (
            f"{predicted_class}. {user_question}"
        )

        document_results = retrieve_from_document(
            document_query,
            document_chunks,
            document_index,
            top_k=3
        )

    report = generate_extended_medical_report(
        top_predictions,
        retrieved_information,
        user_question,
        document_results
    )

    return report

print("Image and document RAG pipeline created successfully.")

Image and document RAG pipeline created successfully.


**Test Without An Uploaded Document**

In [155]:
import  pandas as pd
test_df = pd.read_csv(
    project_path + "/data/processed/test.csv"
)

sample_image_path = test_df.iloc[0]["image_path"]

test_report = run_pipeline_with_document(
    image_path=sample_image_path,
    user_question="What is the risk level and recommended next action?"
)

print(test_report)


SKIN LESION ANALYSIS REPORT

Predicted Lesion:
Melanocytic Nevi

Top-3 Predictions:
- Melanocytic Nevi: 86.53%
- Dermatofibroma: 7.45%
- Actinic Keratoses and Intraepithelial Carcinoma: 2.37%


User Question:
What is the risk level and recommended next action?

Description:
Melanocytic nevi are common moles formed by groups of pigment-producing skin cells.

Typical Signs:
A small, round or oval spot that is usually pink, tan or brown with a clear border.

Risk Level:
Low — most common moles are benign, but unusual or changing moles may require examination.

Recommended Action:
Consult a dermatologist if the mole changes in size, shape, color, sensation, or begins itching or bleeding.

Uploaded Document Information:
No additional document was uploaded.

Supporting Knowledge Source:
National Cancer Institute and American Academy of Dermatology

Medical Disclaimer:
This system is for educational purposes only and is not a replacement
for diagnosis, treatment or advice from a qualified de

**Create A Temporary Test Document**

In [156]:
sample_document_path = (
    documents_path + "/sample_uploaded_document.txt"
)

sample_document_text = """
The patient reports that the skin lesion has slowly changed in size.
The lesion sometimes feels itchy but has not repeatedly bled.
A professional dermatologist examination has been recommended.
"""

with open(
    sample_document_path,
    "w",
    encoding="utf-8"
) as file:
    file.write(sample_document_text)

print("Temporary test document created successfully.")

Temporary test document created successfully.


**Test With Image, Question And Document**

In [157]:
extended_report = run_pipeline_with_document(
    image_path=sample_image_path,
    user_question="Does the uploaded document mention any warning signs?",
    document_path=sample_document_path
)

print(extended_report)


SKIN LESION ANALYSIS REPORT

Predicted Lesion:
Melanocytic Nevi

Top-3 Predictions:
- Melanocytic Nevi: 86.53%
- Dermatofibroma: 7.45%
- Actinic Keratoses and Intraepithelial Carcinoma: 2.37%


User Question:
Does the uploaded document mention any warning signs?

Description:
Melanocytic nevi are common moles formed by groups of pigment-producing skin cells.

Typical Signs:
A small, round or oval spot that is usually pink, tan or brown with a clear border.

Risk Level:
Low — most common moles are benign, but unusual or changing moles may require examination.

Recommended Action:
Consult a dermatologist if the mole changes in size, shape, color, sensation, or begins itching or bleeding.

Uploaded Document Information:
Document Section 1:
The patient reports that the skin lesion has slowly changed in size. The lesion sometimes feels itchy but has not repeatedly bled. A professional dermatologist examination has been recommended.

Supporting Knowledge Source:
National Cancer Institute an

# Phase 4: NLP Medical Report Generator + FAISS RAG Completed ✅

In this phase, we created a local medical knowledge base, connected it with the frozen MobileNetV2 model, and added support for user questions and uploaded documents.

## Medical Knowledge Base

Medical information was added for all seven HAM10000 classes:

* Actinic Keratoses and Intraepithelial Carcinoma
* Basal Cell Carcinoma
* Benign Keratosis-like Lesions
* Dermatofibroma
* Melanoma
* Melanocytic Nevi
* Vascular Lesions

Each entry contains:

* Description
* Typical signs
* Risk level
* Recommended action
* Supporting source

The knowledge base was saved at:

* `rag_data/documents/skin_lesion_knowledge.json`

## FAISS Retrieval System

The medical documents were converted into numerical embeddings using:

* `SentenceTransformer("all-MiniLM-L6-v2")`

The embeddings were stored inside a FAISS similarity-search index.

The FAISS index was saved at:

* `rag_data/index/skin_lesion_faiss.index`

FAISS successfully retrieves the most relevant medical information for the model’s predicted lesion.

## MobileNetV2 Integration

The primary frozen MobileNetV2 model was loaded from:

* `models/mobilenet/mobilenetv2_model.keras`

The model provides:

* Primary predicted lesion
* Top-3 lesion predictions
* Probability percentage for each prediction

Single-image inference was performed on the CPU because GPU prediction caused a cuDNN autotuner error.

## User Question And Document RAG Extension

The system now supports:

* A skin-lesion image
* An optional user question
* An optional PDF, DOCX, or TXT document

Uploaded documents are processed using:

* `pypdf` for PDF files
* `python-docx` for Word files
* Standard Python file reading for TXT files

The document text is:

```text
Extracted
→ divided into smaller chunks
→ converted into embeddings
→ stored in a temporary FAISS index
→ searched using the predicted lesion and user question
```

The document information is used only as supporting context and does not replace professional medical assessment.

## Complete Phase 4 Pipeline

The completed pipeline is:

```text
Skin image
→ MobileNetV2 top-3 predictions
→ Local FAISS medical retrieval
→ Optional document retrieval
→ User-question processing
→ Structured educational medical report
```

The final report contains:

* Predicted lesion
* Top-3 probabilities
* User question
* Lesion description
* Typical signs
* Risk level
* Recommended action
* Relevant uploaded-document information
* Supporting medical source
* Medical disclaimer

The pipeline works both:

* Without an uploaded document
* With an uploaded PDF, DOCX, or TXT document

The main extended pipeline function is:

* `run_pipeline_with_document()`

The system is for educational purposes only and is not a replacement for diagnosis, treatment, or advice from a qualified dermatologist.

Therefore, Phase 4 and its User Document RAG extension are completed successfully.
